Assignment 1 – Feature Engineering

Index Number: 23001951

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

data = {
    "Customer_ID": [
        "C01", "C02", "C03", "C04",
        "C05", "C06", "C07", "C08"
    ],
    "Total_Spent": [
        6000, 12000, 4500, 20000,
        9000, 30000, 8000, 40000
    ],
    "Number_of_Orders": [3, 4, 3, 5, 3, 6, 2, 5],
    "Days_Since_Last_Order": [5, 12, 20, 3, 15, 8, 25, 2],
    "Preferred_Category": [
        "Clothing", "Books", "Clothing", "Electronics",
        "Books", "Electronics", "Clothing", "Books"
    ],
    "Membership_Level": [
        "Bronze", "Silver", "Bronze", "Gold",
        "Silver", "Gold", "Bronze", "Silver"
    ]
}

df = pd.DataFrame(data)

# The provided split: first 6 rows for training, last 2 for testing.
train_df = df.iloc[:6].copy()
test_df = df.iloc[6:].copy()

print("Training dataset shape:", train_df.shape)
print("Test dataset shape:", test_df.shape)

Training dataset shape: (6, 6)
Test dataset shape: (2, 6)


Part A

In [8]:
# Average Order Value = Total Spent / Number of Orders
train_df["Average_Order_Value"] = train_df["Total_Spent"] / train_df["Number_of_Orders"]
test_df["Average_Order_Value"] = test_df["Total_Spent"] / test_df["Number_of_Orders"]

display(train_df[["Customer_ID", "Total_Spent", "Number_of_Orders", "Average_Order_Value"]])
display(test_df[["Customer_ID", "Total_Spent", "Number_of_Orders", "Average_Order_Value"]])

,Customer_ID,Total_Spent,Number_of_Orders,Average_Order_Value
0,C01,6000,3,2000.0
1,C02,12000,4,3000.0
2,C03,4500,3,1500.0
3,C04,20000,5,4000.0
4,C05,9000,3,3000.0
5,C06,30000,6,5000.0


,Customer_ID,Total_Spent,Number_of_Orders,Average_Order_Value
6,C07,8000,2,4000.0
7,C08,40000,5,8000.0


Part B

In [9]:
# B1. Apply base-10 logarithm to Total_Spent in both datasets.
train_df["Log_Total_Spent"] = np.log10(train_df["Total_Spent"])
test_df["Log_Total_Spent"] = np.log10(test_df["Total_Spent"])

# B2. Fit MinMaxScaler on training data only, then transform both datasets.
scaler = MinMaxScaler()
train_df["Days_Since_Last_Order_Scaled"] = scaler.fit_transform(
    train_df[["Days_Since_Last_Order"]]
)
test_df["Days_Since_Last_Order_Scaled"] = scaler.transform(
    test_df[["Days_Since_Last_Order"]]
)

display(train_df[["Customer_ID", "Log_Total_Spent", "Days_Since_Last_Order_Scaled"]])
display(test_df[["Customer_ID", "Log_Total_Spent", "Days_Since_Last_Order_Scaled"]])

,Customer_ID,Log_Total_Spent,Days_Since_Last_Order_Scaled
0,C01,3.778151,0.117647
1,C02,4.079181,0.529412
2,C03,3.653213,1.000000
3,C04,4.301030,0.000000
4,C05,3.954243,0.705882
5,C06,4.477121,0.294118


,Customer_ID,Log_Total_Spent,Days_Since_Last_Order_Scaled
6,C07,3.90309,1.294118
7,C08,4.60206,-0.058824


Part C

In [10]:
# C1. Ordered encoding for membership level.
membership_map = {"Bronze": 0, "Silver": 1, "Gold": 2}
train_df["Membership_Encoded"] = train_df["Membership_Level"].map(membership_map)
test_df["Membership_Encoded"] = test_df["Membership_Level"].map(membership_map)

# C2. One-hot encode Preferred_Category with integer values.
train_categories = pd.get_dummies(
    train_df["Preferred_Category"], prefix="Category", dtype=int
)
test_categories = pd.get_dummies(
    test_df["Preferred_Category"], prefix="Category", dtype=int
)

# C3. Align test columns with training columns, filling absent categories with 0.
test_categories = test_categories.reindex(columns=train_categories.columns, fill_value=0)

# Join the one-hot encoded columns to each dataset.
train_df = pd.concat([train_df, train_categories], axis=1)
test_df = pd.concat([test_df, test_categories], axis=1)

category_columns = list(train_categories.columns)
print("One-hot category columns:", category_columns)
display(train_df[["Customer_ID", "Membership_Level", "Membership_Encoded", "Preferred_Category"] + category_columns])
display(test_df[["Customer_ID", "Membership_Level", "Membership_Encoded", "Preferred_Category"] + category_columns])

One-hot category columns: ['Category_Books', 'Category_Clothing', 'Category_Electronics']


,Customer_ID,Membership_Level,Membership_Encoded,Preferred_Category,Category_Books,Category_Clothing,Category_Electronics
0,C01,Bronze,0,Clothing,0,1,0
1,C02,Silver,1,Books,1,0,0
2,C03,Bronze,0,Clothing,0,1,0
3,C04,Gold,2,Electronics,0,0,1
4,C05,Silver,1,Books,1,0,0
5,C06,Gold,2,Electronics,0,0,1


,Customer_ID,Membership_Level,Membership_Encoded,Preferred_Category,Category_Books,Category_Clothing,Category_Electronics
6,C07,Bronze,0,Clothing,0,1,0
7,C08,Silver,1,Books,1,0,0


Part D

In [11]:
feature_columns = [
    "Number_of_Orders",
    "Average_Order_Value",
    "Log_Total_Spent",
    "Days_Since_Last_Order_Scaled",
    "Membership_Encoded",
] + category_columns

X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

print("X_train:")
display(X_train)
print("X_train shape:", X_train.shape)

print("\nX_test:")
display(X_test)
print("X_test shape:", X_test.shape)

# Confirm that both tables have the same feature columns in the same order.
print("\nSame feature columns and order:", list(X_train.columns) == list(X_test.columns))

X_train:


,Number_of_Orders,Average_Order_Value,Log_Total_Spent,Days_Since_Last_Order_Scaled,Membership_Encoded,Category_Books,Category_Clothing,Category_Electronics
0,3,2000.0,3.778151,0.117647,0,0,1,0
1,4,3000.0,4.079181,0.529412,1,1,0,0
2,3,1500.0,3.653213,1.000000,0,0,1,0
3,5,4000.0,4.301030,0.000000,2,0,0,1
4,3,3000.0,3.954243,0.705882,1,1,0,0
5,6,5000.0,4.477121,0.294118,2,0,0,1


X_train shape: (6, 8)

X_test:


,Number_of_Orders,Average_Order_Value,Log_Total_Spent,Days_Since_Last_Order_Scaled,Membership_Encoded,Category_Books,Category_Clothing,Category_Electronics
6,2,4000.0,3.90309,1.294118,0,0,1,0
7,5,8000.0,4.60206,-0.058824,1,1,0,0


X_test shape: (2, 8)

Same feature columns and order: True


Part E

1) Membership level: Bronze, Silver, and Gold have a natural order from lower to higher membership levels. Therefore, ordered encoding is appropriate: Bronze = 0, Silver = 1, and Gold = 2. These values represent the specified order.

Preferred product category: Clothing, Books, and Electronics are different categories without a natural ranking. One-hot encoding creates a separate binary column for each category, avoiding the false impression that one category is greater than another.

2) The scaler must learn its minimum and maximum values from the training dataset only. The fitted scaler is then used to transform both training and test data. If the test data were included when fitting the scaler, information from the test set would influence preprocessing. This is a form of data leakage and can make model evaluation less reliable.